## Genetic algorithm optimization: turner input/output noise

In this tutorial, we walk through a complete worked example of the genetic
algorithm (GA) optimization mode: we optimize just two parameters of the
`explorer` model's turner module -- its **input noise** and **output
noise** -- so that the simulated turning behavior of `explorer` larvae
resembles the turning behavior recorded in the default reference dataset.

The tutorial produces five videos and three side-by-side comparisons:

1. `video_1_reference.mp4` -- replay of the full reference dataset population (2-segment bodies)
2. `video_2_simulated_dish.mp4` -- a `dish` simulation matched to the reference dataset (same N, same color, `explorer` model)
3. `video_1` vs `video_2`, side by side -- population-level comparison, before optimization
4. `video_3_reference_focused.mp4` -- replay of a single, well-tracked reference larva, camera-focused
5. `video_4_simulated_tethered.mp4` -- a `tethered` simulation of a single `explorer` larva, focused
6. `video_3` vs `video_4`, side by side -- single-larva turning comparison, before optimization
7. GA optimization of the turner module's `input_noise`/`output_noise` against the reference dataset's turning statistics
8. `video_5_simulated_optimized.mp4` -- a final `dish` simulation using the optimized model
9. `video_5` vs `video_1`, side by side -- population-level comparison, after optimization

All simulations use the reference dataset's own duration and timestep
(`dt`), so that every video is directly comparable frame-for-frame.

In [ ]:
%load_ext param.ipython
import time
from pathlib import Path

import larvaworld as lw
from larvaworld.lib import reg, util
from larvaworld.lib.sim import ReplayRun, ExpRun
from larvaworld.lib.sim.genetic_algorithm import GAlauncher
from larvaworld.lib.reg.generators import ReplayConf
from larvaworld.lib.model.modules.module_modes import moduleDB, class_objs
from larvaworld.lib.util.combining import combine_videos

lw.VERBOSE = 1

# Tutorial safety switches (avoid heavy compute / media generation by default)
RUN_VIDEO_DEMOS = False
RUN_GA_DEMO = False

MEDIA_DIR = "./media/ga_turner_noise"
BESTCONF_ID = "explorer_optimized_turner"

# The GA optimization loop itself uses a shorter proxy duration for fitness
# evaluation (standard practice -- optimizing against the full multi-minute
# recording for every genome of every generation would be prohibitively
# slow). The five comparison videos below all use the reference dataset's
# full, matched duration/dt so they remain directly comparable.
DEMO_GA_DURATION_MIN = 0.3
DEMO_GA_NAGENTS = 8
DEMO_GA_NELITS = 2
DEMO_GA_NGENERATIONS = 2

### Step 1 -- Reference dataset replay (full population)

Load the default reference dataset and replay it in full, reconstructing
each larva's body with 2 segments.

In [ ]:
refID = reg.default_refID
d = reg.loadRef(refID)
d.load()

N = d.config.N
color = d.config.color
dt = d.config.dt
duration_min = d.config.duration

print(f"Reference dataset: {refID}")
print(f"N={N} agents, color={color!r}, dt={dt}s, duration={duration_min:.2f} min")

In [ ]:
if RUN_VIDEO_DEMOS:
    p1 = ReplayConf(refID=refID, draw_Nsegs=2).nestedConf
    screen_kws1 = {
        "show_display": False,
        "vis_mode": "video",
        "save_video": True,
        "media_dir": MEDIA_DIR,
        "video_file": "video_1_reference",
    }
    rep1 = ReplayRun(
        parameters=p1,
        id=f"{refID}_replay_full",
        dir=f"{MEDIA_DIR}/rep1",
        screen_kws=screen_kws1,
    )
    rep1.run()

### Step 2 -- Simulated dish (full population, matched)

Run a `dish` experiment with the same number of agents and the same larva
color as the reference dataset, using the `explorer` model (the `dish`
experiment's default larva model), and the reference dataset's own
duration and timestep.

In [ ]:
dish_conf = reg.conf.Exp.getID("dish")
dish_conf.larva_groups.explorer.distribution.N = N
dish_conf.larva_groups.explorer.color = color
print(dish_conf.larva_groups.explorer.model.brain.turner)

if RUN_VIDEO_DEMOS:
    screen_kws2 = {
        "show_display": False,
        "vis_mode": "video",
        "save_video": True,
        "media_dir": MEDIA_DIR,
        "video_file": "video_2_simulated_dish",
    }
    r2 = ExpRun(
        parameters=dish_conf,
        duration=duration_min,
        dt=dt,
        screen_kws=screen_kws2,
        store_data=False,
    )
    r2.simulate()

### Step 3 -- Side-by-side #1: reference vs. simulated dish

Combine the two population-level videos side by side: reference on the
left, simulated on the right.

In [ ]:
if RUN_VIDEO_DEMOS:
    combine_videos(
        file_dir=MEDIA_DIR,
        save_as="video_side_by_side_1_reference_vs_dish.mp4",
        files=[
            f"{MEDIA_DIR}/video_1_reference.mp4",
            f"{MEDIA_DIR}/video_2_simulated_dish.mp4",
        ],
    )

*[video_side_by_side_1_reference_vs_dish.mp4](media/ga_turner_noise/video_side_by_side_1_reference_vs_dish.mp4)*

At the population level, the two videos are expected to differ mainly in
**turning behavior**: body bend and orientation dynamics of the
`explorer` model's default (unoptimized) turner are not tuned against
this reference dataset, so the simulated larvae's turning pattern may not
resemble the real one yet -- this is exactly what the GA optimization
below addresses.

### Step 4 -- Focused replay of a single reference larva

To examine turning behavior more closely, replay a single, well-tracked
larva from the reference dataset: the one with the fewest position-data
gaps (i.e. the longest continuous visible trajectory). The camera stays
focused on this larva throughout, again with a 2-segment body.

In [ ]:
xy_cols = util.nam.xy(d.config.point)
nan_counts = d.s[xy_cols[0]].isna().groupby(level="AgentID").sum()
best_agent_id = nan_counts.idxmin()
best_agent_idx = list(d.agent_ids).index(best_agent_id)
print(f"Most-complete trajectory: {best_agent_id} ({nan_counts.min()} missing samples)")

if RUN_VIDEO_DEMOS:
    p4 = ReplayConf(
        refID=refID,
        agent_ids=[best_agent_idx],
        close_view=True,
        fix_point=6,
        draw_Nsegs=2,
    ).nestedConf
    screen_kws4 = {
        "show_display": False,
        "vis_mode": "video",
        "save_video": True,
        "media_dir": MEDIA_DIR,
        "video_file": "video_3_reference_focused",
    }
    rep4 = ReplayRun(
        parameters=p4,
        id=f"{refID}_replay_focused",
        dir=f"{MEDIA_DIR}/rep4",
        screen_kws=screen_kws4,
    )
    rep4.run()

### Step 5 -- Tethered simulation of a single larva

Run a `tethered` experiment (single, spatially fixed `explorer` larva --
the display is naturally focused since there is only one agent, held at
the arena center) with the same matched duration and timestep.

In [ ]:
tethered_conf = reg.conf.Exp.getID("tethered")

if RUN_VIDEO_DEMOS:
    screen_kws5 = {
        "show_display": False,
        "vis_mode": "video",
        "save_video": True,
        "media_dir": MEDIA_DIR,
        "video_file": "video_4_simulated_tethered",
    }
    r5 = ExpRun(
        parameters=tethered_conf,
        modelIDs=["explorer"],
        duration=duration_min,
        dt=dt,
        screen_kws=screen_kws5,
        store_data=False,
    )
    r5.simulate()

### Step 6 -- Side-by-side #2: reference vs. simulated tethered, focused

In [ ]:
if RUN_VIDEO_DEMOS:
    combine_videos(
        file_dir=MEDIA_DIR,
        save_as="video_side_by_side_2_reference_vs_tethered.mp4",
        files=[
            f"{MEDIA_DIR}/video_3_reference_focused.mp4",
            f"{MEDIA_DIR}/video_4_simulated_tethered.mp4",
        ],
    )

*[video_side_by_side_2_reference_vs_tethered.mp4](media/ga_turner_noise/video_side_by_side_2_reference_vs_tethered.mp4)*

Focusing on a single larva makes the turning-behavior mismatch easier to
see directly: the simulated animal's turning is unlikely to resemble the
real animal's turning yet, since the `explorer` model's turner has not
been fit to this dataset.

### Step 7 -- GA optimization setup: turner input/output noise only

We now set up a GA optimization that adjusts **exclusively** the turner
module's `input_noise` and `output_noise` -- no other turner parameter is
included in the search space -- starting from the `explorer` model
configuration, with fitness based on how closely the simulated turning
behavior matches the reference dataset.

We base the setup on the `"exploration"` GA experiment preset: it already
targets this reference dataset and already weighs turning-related
(`"angular kinematics"`) evaluation metrics, and its `dt` already matches
the reference dataset's own timestep.

`GAselector`/`SpaceDict`'s normal `space_mkeys` mechanism builds the
optimization space from an *entire* module (e.g. `space_mkeys=["turner"]`
would include every turner parameter), and deliberately excludes
`Effector`-level parameters like `input_noise`/`output_noise` from that
per-module search space. To restrict the search to exactly these two
parameters, we build their parameter objects directly from the turner's
mode class and assign them onto the selector's optimization space after
construction.

In [ ]:
p = reg.conf.Ga.expand("exploration")
assert p.dt == dt  # already matches the reference dataset's timestep

p.ga_select_kws.base_model = "explorer"
p.ga_select_kws.bestConfID = BESTCONF_ID
p.ga_select_kws.space_mkeys = []  # narrowed manually below
p.ga_select_kws.Nagents = DEMO_GA_NAGENTS
p.ga_select_kws.Nelits = DEMO_GA_NELITS
p.ga_select_kws.Ngenerations = DEMO_GA_NGENERATIONS

ga = GAlauncher(
    parameters=p, duration=DEMO_GA_DURATION_MIN, screen_kws={"show_display": False}
)

# Build the turner-noise-only optimization space.
mConf0 = reg.conf.Model.getID("explorer")
turner_conf = mConf0.brain.turner
turner_mode_class = moduleDB.BrainModuleModes["turner"][turner_conf.mode]
noise_objs = class_objs(turner_mode_class, excluded=["phi", "name"])
wanted = {k: v for k, v in noise_objs.items() if k in ("input_noise", "output_noise")}

space = util.AttrDict()
for pname, obj in wanted.items():
    if pname in turner_conf:
        obj.default = turner_conf[pname]
    space[f"brain.turner.{pname}"] = obj

ga.selector.space_objs = space
ga.selector.space_ks = space.keylist
ga.selector.parclasses = util.AttrDict(
    {k: ga.selector.parclass(k) for k in ga.selector.space_ks}
)

print("Optimization space:", ga.selector.space_ks)
print("Defaults:", ga.selector.defaults)

### Step 8 -- Run the GA optimization

Run the GA for several generations. The demo settings above
(`DEMO_GA_NAGENTS`, `DEMO_GA_NGENERATIONS`, `DEMO_GA_DURATION_MIN`) keep
this tutorial's default runtime short; increase them for a more thorough
optimization.

In [ ]:
if RUN_GA_DEMO:
    best = ga.simulate()
    print("Best fitness:", ga.best_fitness)
    print("Optimized turner config:", best.mConf.brain.turner)

### Step 9 -- The optimized model is saved automatically

`GAselector.bestConfID` (set to `"explorer_optimized_turner"` above)
causes the GA launcher to register the current best genome's model under
that ID in the model registry every time a new best genome is found --
no separate save step is required.

In [ ]:
if RUN_GA_DEMO:
    print(BESTCONF_ID in reg.conf.Model.confIDs)
    print(reg.conf.Model.getID(BESTCONF_ID).brain.turner)

### Step 10 -- Final simulation with the optimized model

Run a new `dish` simulation, matched in population size to the reference
dataset as before, this time using the `explorer_optimized_turner` model.

In [ ]:
if RUN_VIDEO_DEMOS and RUN_GA_DEMO:
    screen_kws10 = {
        "show_display": False,
        "vis_mode": "video",
        "save_video": True,
        "media_dir": MEDIA_DIR,
        "video_file": "video_5_simulated_optimized",
    }
    r10 = ExpRun(
        parameters=dish_conf,
        modelIDs=[BESTCONF_ID],
        duration=duration_min,
        dt=dt,
        screen_kws=screen_kws10,
        store_data=False,
    )
    r10.simulate()

### Step 11 -- Final side-by-side: optimized vs. reference

In [ ]:
if RUN_VIDEO_DEMOS and RUN_GA_DEMO:
    combine_videos(
        file_dir=MEDIA_DIR,
        save_as="video_side_by_side_3_optimized_vs_reference.mp4",
        files=[
            f"{MEDIA_DIR}/video_5_simulated_optimized.mp4",
            f"{MEDIA_DIR}/video_1_reference.mp4",
        ],
    )

*[video_side_by_side_3_optimized_vs_reference.mp4](media/ga_turner_noise/video_side_by_side_3_optimized_vs_reference.mp4)*

### Before vs. after

Comparing the "before" pair (`video_1` vs. `video_2`, unoptimized turner)
with the "after" pair (`video_1` vs. `video_5`, optimized turner) is the
point of this tutorial: ideally, after optimizing just the turner
module's input and output noise against the reference dataset, the
simulated larvae's turning behavior resembles the real animals' turning
behavior more closely than it did with the default `explorer`
configuration -- without having touched any other part of the model.